# Aprendizagem de maquina - aula de 16/09

Notebook de apoio para a apresentacao interativa: supervisionado vs nao supervisionado, scikit-learn, algoritmos classicos, AutoML, redes neurais e metricas.

Objetivo da aula: sair da ideia abstrata de que um modelo aprende e chegar ao ciclo pratico: dados -> treino -> predicao -> avaliacao -> melhoria.

## 1. Imports e datasets sinteticos

Vamos usar bases artificiais pequenas porque elas ajudam a enxergar o comportamento dos algoritmos sem depender de um dominio especifico.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification, make_blobs, make_regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

X_clf, y_clf = make_classification(
    n_samples=500,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.2,
    random_state=RANDOM_STATE,
)

X_reg, y_reg = make_regression(
    n_samples=300,
    n_features=1,
    noise=18,
    random_state=RANDOM_STATE,
)

plt.scatter(X_clf[:, 0], X_clf[:, 1], c=y_clf, cmap="coolwarm", s=18)
plt.title("Problema supervisionado: cada ponto tem classe")
plt.show()

## 2. Supervisionado: treino/teste

No aprendizado supervisionado, o conjunto de treino tem `X` e `y`. O teste fica separado para simular dados novos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf, test_size=0.25, stratify=y_clf, random_state=RANDOM_STATE
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)
print("Proporcao da classe positiva no treino:", y_train.mean().round(2))

## 3. Regressao linear

A regressao linear aprende coeficientes para aproximar uma variavel numerica. Use como baseline interpretavel.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=RANDOM_STATE
)

reg = LinearRegression()
reg.fit(Xr_train, yr_train)
pred_reg = reg.predict(Xr_test)

mae = mean_absolute_error(yr_test, pred_reg)
rmse = mean_squared_error(yr_test, pred_reg, squared=False)
r2 = r2_score(yr_test, pred_reg)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {r2:.2f}")

plt.scatter(Xr_test[:, 0], yr_test, label="real", s=18)
plt.scatter(Xr_test[:, 0], pred_reg, label="predito", s=18)
plt.legend()
plt.title("Regressao linear: real vs predito")
plt.show()

## 4. Classificadores classicos

Vamos comparar k-NN, arvore de decisao, random forest e Naive Bayes usando o mesmo split.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    "k-NN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)),
    "Arvore": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
    "Random forest": RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE),
    "Naive Bayes": GaussianNB(),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({
        "modelo": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
    })

pd.DataFrame(rows).sort_values("f1", ascending=False).round(3)

## 5. Metricas e matriz de confusao

Acuracia responde 'quantos acertei no total?'. Precision responde 'quando eu disse sim, quantas vezes era sim?'. Recall responde 'dos casos sim, quantos encontrei?'. F1 equilibra precision e recall.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

best_model = models["Random forest"]
best_model.fit(X_train, y_train)
pred = best_model.predict(X_test)

print(classification_report(y_test, pred, target_names=["classe 0", "classe 1"]))
ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap="Blues")
plt.title("Matriz de confusao")
plt.show()

## 6. Nao supervisionado: K-Means

Aqui nao passamos `y` para o algoritmo. Ele procura grupos apenas pela estrutura dos pontos.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

X_blob, _ = make_blobs(n_samples=450, centers=3, cluster_std=1.2, random_state=RANDOM_STATE)
kmeans = KMeans(n_clusters=3, n_init="auto", random_state=RANDOM_STATE)
clusters = kmeans.fit_predict(X_blob)

plt.scatter(X_blob[:, 0], X_blob[:, 1], c=clusters, cmap="viridis", s=18)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], c="red", marker="x", s=130)
plt.title("K-Means: grupos descobertos sem rotulo")
plt.show()

pca = PCA(n_components=1)
X_reduzido = pca.fit_transform(X_blob)
print("Variancia explicada por 1 componente:", pca.explained_variance_ratio_[0].round(3))

## 7. Rede neural simples com scikit-learn

O `MLPClassifier` mostra a ideia de camadas densas. Para deep learning moderno em imagem, audio ou texto, normalmente usamos PyTorch, TensorFlow ou Keras.

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=800, random_state=RANDOM_STATE),
)

scores = cross_val_score(mlp, X_clf, y_clf, cv=5, scoring="f1")
print("F1 medio em validacao cruzada:", scores.mean().round(3))
print("Desvio:", scores.std().round(3))

## 8. AutoML: qual biblioteca usar?

Sugestao para a aula: **AutoGluon Tabular** quando o foco for desempenho em dados tabulares com poucas linhas de codigo. Alternativa leve: **FLAML**, boa para mostrar busca eficiente com limite de tempo.

Use esta parte como celula opcional, porque AutoML pode instalar dependencias mais pesadas.

In [ ]:
# Opcao A: AutoGluon Tabular
# Instale antes, se necessario: pip install autogluon.tabular
#
# from autogluon.tabular import TabularPredictor
# df = pd.DataFrame(X_clf, columns=["x1", "x2"])
# df["alvo"] = y_clf
# train_df, test_df = train_test_split(df, test_size=0.25, stratify=df["alvo"], random_state=RANDOM_STATE)
# predictor = TabularPredictor(label="alvo", eval_metric="f1").fit(train_df, time_limit=60)
# predictor.leaderboard(test_df)

# Opcao B: FLAML
# Instale antes, se necessario: pip install flaml[automl]
#
# from flaml import AutoML
# automl = AutoML()
# automl.fit(X_train, y_train, task="classification", time_budget=30, metric="f1")
# print(automl.best_estimator, automl.best_config)
# print("F1:", f1_score(y_test, automl.predict(X_test)))

## 9. Fechamento para discussao

- Qual modelo voce escolheria se precisasse explicar a decisao?
- Qual modelo voce escolheria se o objetivo fosse apenas desempenho tabular?
- O que muda quando a metrica principal deixa de ser acuracia e passa a ser recall?
- Que risco aparece quando o modelo vai muito bem no treino e mal no teste?